# 🔥 Chapter 7 — The Training Loop: Loss, Optimizers & Training

---

### 🎯 What you'll learn
- Loss functions: when to use which
- Optimizers: SGD, Adam, AdamW
- The complete training loop pattern
- Validation loop and overfitting detection
- ⚠️ Corner cases: forgetting `model.train()`, gradient clipping, learning rate

## 7.1 — Loss Functions

In [ ]:
import torch
import torch.nn as nn

# ── Regression ──
mse = nn.MSELoss()       # Mean Squared Error
mae = nn.L1Loss()         # Mean Absolute Error

pred = torch.tensor([2.5, 3.0, 4.5])
true = torch.tensor([3.0, 3.0, 4.0])
print(f"MSE: {mse(pred, true):.4f}")
print(f"MAE: {mae(pred, true):.4f}")

# ── Binary Classification ──
bce = nn.BCEWithLogitsLoss()  # Includes sigmoid — more numerically stable!
logits = torch.tensor([0.5, -1.0, 2.0])
labels = torch.tensor([1.0, 0.0, 1.0])
print(f"\nBCE: {bce(logits, labels):.4f}")
# ⚠️ Use BCEWithLogitsLoss, NOT BCELoss + sigmoid separately!

# ── Multi-class Classification ──
ce = nn.CrossEntropyLoss()  # Includes softmax internally!
logits = torch.randn(3, 5)          # 3 samples, 5 classes
labels = torch.tensor([0, 3, 2])    # Class indices (NOT one-hot!)
print(f"CrossEntropy: {ce(logits, labels):.4f}")

# ⚠️ Corner Case: CrossEntropyLoss expects:
#   - Input: raw logits (N, C), NOT softmax probabilities!
#   - Target: class INDICES (N,), NOT one-hot vectors!

## 7.2 — Optimizers

In [ ]:
model = nn.Linear(10, 3)

# SGD — simple, needs tuning
optim_sgd = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam — adaptive, great default
optim_adam = torch.optim.Adam(model.parameters(), lr=0.001)

# AdamW — Adam with decoupled weight decay (preferred for transformers)
optim_adamw = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# ⚠️ Corner Case: Adam vs AdamW weight decay
# Adam's weight_decay is coupled with the adaptive learning rate — this is wrong!
# AdamW decouples it — always prefer AdamW when using weight decay.
print("Prefer AdamW over Adam when using weight_decay!")

## 7.3 — The Complete Training Loop

This is the **single most important pattern** in PyTorch.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── Setup ──
torch.manual_seed(42)

# Synthetic data: y = 3x₁ + 2x₂ + 1
X = torch.randn(1000, 2)
y = 3 * X[:, 0] + 2 * X[:, 1] + 1 + 0.1 * torch.randn(1000)
y = y.unsqueeze(1)

# Split
X_train, X_val = X[:800], X[800:]
y_train, y_val = y[:800], y[800:]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

# Model
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# ── Training Loop ──
num_epochs = 20
for epoch in range(num_epochs):
    # ✅ MUST: Set model to training mode
    model.train()
    train_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        # 1. Forward pass
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        
        # 2. Backward pass
        optimizer.zero_grad()   # ⚠️ MUST zero gradients!
        loss.backward()
        
        # 3. Update weights
        optimizer.step()
        
        train_loss += loss.item() * X_batch.size(0)
    
    train_loss /= len(train_loader.dataset)
    
    # ── Validation Loop ──
    model.eval()   # ✅ MUST: Set model to eval mode
    val_loss = 0.0
    with torch.no_grad():   # ✅ MUST: disable gradients for validation
        for X_batch, y_batch in val_loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(val_loader.dataset)
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

print("\n✅ Training complete!")

## 7.4 — ⚠️ Critical Corner Cases

In [ ]:
# ⚠️ Corner Case 1: model.train() vs model.eval()
# model.train() — enables Dropout and updates BatchNorm running stats
# model.eval()  — disables Dropout and freezes BatchNorm
# FORGETTING THIS IS THE #1 CAUSE OF "MY MODEL WORKS IN TRAINING BUT NOT INFERENCE"!

# ⚠️ Corner Case 2: optimizer.zero_grad() placement
# Option A (traditional): before loss.backward()
# Option B (modern): optimizer.zero_grad(set_to_none=True)  ← slightly faster
# set_to_none=True sets .grad to None instead of zeros (less memory)

# ⚠️ Corner Case 3: Gradient Clipping (prevents exploding gradients)
model = nn.Linear(10, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

x = torch.randn(1, 10)
loss = model(x).sum()
loss.backward()

# Clip gradient norm to max 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
print("Gradient clipping applied!")

# ⚠️ Corner Case 4: Moving model to device BEFORE creating optimizer
# model = model.to(device)         ← Do this FIRST
# optimizer = Adam(model.parameters())  ← THEN create optimizer
# If reversed, optimizer tracks CPU parameters but model is on GPU!

## ✏️ Exercises

In [ ]:
# Exercise 1: Modify the training loop above to:
# a) Add early stopping (stop if val_loss hasn't improved for 5 epochs)
# b) Save the best model weights
# YOUR CODE HERE


# Exercise 2: Train a model on synthetic classification data:
# 2 features, 3 classes. Use CrossEntropyLoss.
# YOUR CODE HERE
